# FSDP(分片存储、按需加载)


FSDP（Fully Sharded Data Parallel，全分片数据并行）是 PyTorch 原生提供的一种高效分布式训练方案，专门用来解决大模型训练中遇到的“显存墙”问题。

- 简单来说，<font color='red'>传统的分布式数据并行（DDP）要求每张 GPU 上都存放一份完整的模型副本</font>，当模型达到几十亿甚至上百亿参数时，单张显卡的显存根本装不下。
- 而 FSDP 的核心思想就是“<font color='red'>分片存储、按需加载</font>”，它可以让你用有限的显卡资源去训练远超单卡显存限制的超大模型。

## 💡 FSDP 的核心原理：化整为零
FSDP 的工作流程可以形象地比喻为“多人合作组装一个巨型乐高模型”：
1. <b>分片存储（初始化）</b>：
    - <font color='red'>FSDP 会将模型的参数、梯度以及优化器状态（如 Adam 的动量）切分成多份，均匀地分散存储在各个 GPU 上。平时闲置时，每张卡只保管自己那一小部分“乐高零件”。</font>
2. <b>按需聚合（前向/反向传播</b>）：<font color='green'>当 GPU 需要计算某一层网络时，它会通过高效的集合通信操作（All-Gather），快速从其他 GPU 那里把这一层缺失的零件“借”过来，在本地临时拼出完整的层参数进行计算。</font>
3. <b>即用即弃（释放显存</b>）：<font color='red'>这一层计算一结束，GPU 会立刻把借来的零件丢弃（释放显存），只保留自己原本负责的那部分分片，然后继续处理下一层。</font>
4. <b>梯度同步</b>：在反向传播计算完梯度后，FSDP 会<font color='red'>使用 Reduce-Scatter 操作，将梯度汇总并平均分发给各个 GPU，确保每张卡只更新自己负责的那部分参数。</font>

## 📊 FSDP 与传统 DDP 的对比
为了更直观地理解 FSDP 的优势，我们可以通过下表进行对比：

| 特性 | DDP (传统数据并行) | FSDP (全分片数据并行) |
| :--- | :--- | :--- |
| 参数存储 | <font color='red'>每张 GPU 存储完整模型副本</font> | <font color='red'>模型参数被切分，分散在各卡</font> |
| 显存占用 | 极高（受限于单卡最大显存） | 极低（理论上随显卡数量线性下降） |
| 通信方式 | <font color='red'>训练结束后一次性同步梯度 (All-Reduce)</font> | <font color='red'>计算时频繁通信聚合参数</font> (All-Gather / Reduce-Scatter) |
| 适用场景 | 中小规模模型 | 数十亿至千亿级参数的大模型 |

## ⚙️ FSDP 的关键策略与配置
在实际使用中，FSDP 提供了非常灵活的配置选项，让你可以在“显存节省”和“通信开销”之间做权衡：
- 分片策略 (sharding_strategy)：
    - <b>FULL_SHARD（默认，类似 ZeRO-3）</b>：对参数、梯度、优化器状态全部进行分片。<font color='red'>显存节省最多，适合训练最大的模型。</font>
    - <b>SHARD_GRAD_OP（类似 ZeRO-2）</b>：<font color='red'>只分片梯度和优化器状态，模型参数在每张卡上保持完整。通信量较少，适合模型能塞进显存但优化器状态太大的情况。</font>
    - <b>NO_SHARD</b>：不分片，行为等同于传统的 DDP。
    
- <font color='red'>混合精度训练 (mixed_precision)</font>：
    - 配合 bfloat16 或 float16，不仅能将显存占用减半，还能利用现代 GPU 的 Tensor Core 极大提升计算速度。
- <font color='red'>CPU 卸载 (cpu_offload)</font>：
    - 如果显存实在不够，FSDP 还可以把暂时不用的参数分片挪到 CPU 内存里。<font color='red'>这是一种“用速度换空间”的策略，虽然会拖慢训练，但能让你在消费级显卡上跑起巨型模型。</font>
- <font color='red'>自动包装 (auto_wrap_policy)</font>：<font color='green'>FSDP 不需要你手动去给每一层网络写分片代码。你可以设置策略（比如按层的大小），让 FSDP 自动识别并包装 Transformer 的大模块（如 DecoderLayer），在通信效率和计算粒度之间取得最佳平衡。</font>

## 💻 极简代码示例
在 PyTorch 中启用 FSDP 非常简单，通常只需要用 FSDP() 包装你的模型即可：

In [ ]:
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
from torch.distributed.fsdp.api import ShardingStrategy, MixedPrecision
import torch

# 1. 定义自动包装策略（例如：只对参数量大于 1亿 的模块进行分片）
my_auto_wrap_policy = size_based_auto_wrap_policy(min_num_params=1e8)

# 2. 定义混合精度策略（推荐在 A100/H100 等现代显卡上使用 bfloat16）
mixed_precision_policy = MixedPrecision(
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.bfloat16,
    buffer_dtype=torch.bfloat16,
)

# 3. 使用 FSDP 包装模型
fsdp_model = FSDP(
    model,
    auto_wrap_policy=my_auto_wrap_policy,
    mixed_precision=mixed_precision_policy,
    sharding_strategy=ShardingStrategy.FULL_SHARD, # 全分片模式
    device_id=torch.cuda.current_device()
)

## 总的来说:
- FSDP 是 PyTorch 生态中训练大模型的工业级标准方案。它既保留了 DDP 的易用性，又通过精细化的内存管理打破了显存瓶颈。如果你在训练大模型时遇到了显存不足（OOM）的问题，FSDP 通常是首选的优化方案。